In [1]:
%matplotlib qt

import h5py
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.ticker as ticker

from scipy.interpolate import RegularGridInterpolator
from numpy.polynomial.chebyshev import Chebyshev
from numpy.polynomial import Polynomial

In [2]:
#Cargar datos
LABELS = ["Alt_Si3N4", "Anc_Si3N4", "lambda", "n_eff", "A_eff"]

def load_and_normalize(fname):
    with h5py.File(fname, "r") as f:
        A = f["/A"][...]
    axis5 = next((ax for ax, s in enumerate(A.shape) if s == 5), None)
    if axis5 is None:
        raise ValueError(f"No encontré eje de tamaño 5 en {fname}")
    if axis5 != A.ndim - 1:
        A = np.moveaxis(A, axis5, -1)
    return A

In [3]:
archivos = [
    "Datos_reto/Datos9.h5",
    "Datos_reto/Datos11.h5"
]
datasets = [load_and_normalize(fname) for fname in archivos]

def check_borders(d1, d2):
    alt1_end = np.round(d1[-1,0,0,0],6)
    alt2_start = np.round(d2[0,0,0,0],6)
    anc1_end = np.round(d1[0,-1,0,1],6)
    anc2_start = np.round(d2[0,0,0,1],6)
    print(f"Check Alt: {alt1_end} vs {alt2_start}, Check Anc: {anc1_end} vs {anc2_start}")
    if not np.isclose(alt1_end, alt2_start) and not np.isclose(anc1_end, anc2_start):
        print("⚠️ Advertencia: los bordes no coinciden perfectamente")

for i in range(len(datasets)-1):
    check_borders(datasets[i], datasets[i+1])

DatosU = np.concatenate(datasets, axis=0)

Check Alt: 0.4 vs 0.4, Check Anc: 2.0 vs 0.5


In [5]:
with h5py.File("Datos_reto/DatosU.h5", "w") as f_out:
    f_out.create_dataset("A", data=DatosU)

print("✅ Dataset unificado guardado en DatosU.h5, shape:", DatosU.shape)

✅ Dataset unificado guardado en DatosU.h5, shape: (100, 10, 10, 5)


In [10]:
def to_dataframe(arr):
    nA, nB, nC, _ = arr.shape
    rows = []
    for ii in range(nA):
        for jj in range(nB):
            for kk in range(nC):
                rec = {"i": ii+1, "j": jj+1, "k": kk+1}
                for p, name in enumerate(LABELS):
                    rec[name] = arr[ii, jj, kk, p]
                rows.append(rec)
    return pd.DataFrame(rows, columns=["i","j","k"]+LABELS)

df = to_dataframe(DatosU)
df["lambda_nm"] = df["lambda"] * 1000 

df2 = df.drop(columns=['i','j','k','A_eff','lambda_nm'])
print(df2)

df2.to_csv('datostsv.tsv', sep="\t")

#Guarda todos los datos en un dataframe y donde i: lambdas, j: anchuras, k: alturas

      Alt_Si3N4  Anc_Si3N4  lambda     n_eff
0      0.400000        0.5   1.035  1.594476
1      0.466667        0.5   1.035  1.627080
2      0.533333        0.5   1.035  1.652778
3      0.600000        0.5   1.035  1.672627
4      0.666667        0.5   1.035  1.688365
...         ...        ...     ...       ...
9995   0.733333        2.0   1.035  1.920578
9996   0.800000        2.0   1.035  1.930293
9997   0.866667        2.0   1.035  1.938219
9998   0.933333        2.0   1.035  1.944864
9999   1.000000        2.0   1.035  1.950377

[10000 rows x 4 columns]


In [6]:
#Grafica N_efectivo
fig1 = plt.figure(figsize=(10,8))
ax1 = fig1.add_subplot(111, projection='3d')
p1 = ax1.scatter(df["Alt_Si3N4"], df["Anc_Si3N4"], df["lambda"],
                 c=df["n_eff"], cmap="viridis", s=10)
ax1.set_xlabel("Altura (µm)")
ax1.set_ylabel("Ancho (µm)")
ax1.set_zlabel("Lambda (µm)")
fig1.colorbar(p1, ax=ax1, label="n_eff")
ax1.set_title("n_eff")

#Grafica Area efectiva
"""
fig2 = plt.figure(figsize=(10,8))
ax2 = fig2.add_subplot(111, projection='3d')
p2 = ax2.scatter(df["Alt_Si3N4"], df["Anc_Si3N4"], df["lambda"],
                 c=df["A_eff"], cmap="plasma", s=10)
ax2.set_xlabel("Altura (µm)")
ax2.set_ylabel("Ancho (µm)")
ax2.set_zlabel("Lambda (µm)")
fig2.colorbar(p2, ax=ax2, label="A_eff")
ax2.set_title("A_eff")
"""
plt.show()

In [28]:
#Para una anchura j seleccionada y una altura k seleccionada del dataframe, interpola con un grado dado y devuelve los 
#coeficientes del polinomio
def ajustar_neff_vs_lambda(df, grado=30):
    if len(df) <= grado:
        raise ValueError("No hay suficientes puntos para ajuste polinomial.")

    x = df["lambda"].values
    y = df["n_eff"].values

    coef = np.polyfit(x, y, grado)
    p = np.poly1d(coef)
    return p

def ajustar_neff_vs_lambda_cheb(df, grado=30):
    x = df["lambda"].values
    y = df["n_eff"].values

    # Ajuste estable en la base de Chebyshev; 'domain' reescala x a [-1,1]
    dom = [x.min(), x.max()]
    cheb = Chebyshev.fit(x, y, deg=grado, domain=dom)  # evita mal condicionamiento

    # Devolver objeto evaluable + derivada exacta
    def neff(xq):   return cheb(xq)
    def dneff(xq):  return cheb.deriv()(xq)
    return cheb, neff, dneff

In [29]:
print("Rango de λ en el dataset:", df["lambda"].min(), "–", df["lambda"].max())
print("Número de λ únicos:", len(df["lambda"].unique()))

Rango de λ en el dataset: 0.5 – 1.57
Número de λ únicos: 99


In [30]:
#Se prueba de que se estan escogiendo correctamente las geometrias
df_geom = df[(df["j"]==3) & (df["k"]==4)]

df_geom["lambda"] = df_geom["lambda"].round(6)
df_geom = df_geom.drop_duplicates(subset="lambda", keep="first").sort_values("lambda")
lam_data = df_geom["lambda"].values
neff_data = df_geom["n_eff"].values

print(df_geom[["lambda","n_eff"]])
print(len(df_geom[["lambda","n_eff"]]))

        lambda     n_eff
5023  0.500000  2.010774
5123  0.510918  2.005907
5223  0.521837  2.001181
5323  0.532755  1.996579
5423  0.543673  1.992089
...        ...       ...
4523  1.526327  1.628556
4623  1.537245  1.624135
4723  1.548163  1.619712
4823  1.559082  1.615288
4923  1.570000  1.610864

[99 rows x 2 columns]
99


C:\Users\david\AppData\Local\Temp\ipykernel_29252\1962944012.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_geom["lambda"] = df_geom["lambda"].round(6)


In [ ]:
j_sel, k_sel = 3, 1

df_geom = df[(df["j"]==j_sel) & (df["k"]==k_sel)]

df_geom["lambda"] = df_geom["lambda"].round(6)
df_geom = df_geom.drop_duplicates(subset="lambda", keep="first").sort_values("lambda")
lam_data = df_geom["lambda"].values
neff_data = df_geom["n_eff"].values

alto = df_geom["Alt_Si3N4"].iloc[0]
ancho = df_geom["Anc_Si3N4"].iloc[0]

lambdas = len(df_geom["lambda"].unique())

print(f"Geometría seleccionada: j={j_sel}, k={k_sel}")
print(f"Altura = {alto:.4f} µm")
print(f"Ancho  = {ancho:.4f} µm")
print(f"Hay {lambdas} longitudes de onda")

Geometría seleccionada: j=3, k=1
Altura = 0.4000 µm
Ancho  = 0.8333 µm
Hay 99 longitudes de onda


C:\Users\david\AppData\Local\Temp\ipykernel_29252\2852135929.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_geom["lambda"] = df_geom["lambda"].round(6)


In [32]:
c = 2.99792458e14  # m/s
p = ajustar_neff_vs_lambda(df, grado=30)

cheb, neff, dneff = ajustar_neff_vs_lambda_cheb(df_geom)

def polinomio_a_str(p):
    terms = []
    for i, coef in enumerate(p.coefficients):
        exp = p.order - i
        if exp == 0:
            terms.append(f"{coef:+.4f}")
        elif exp == 1:
            terms.append(f"{coef:+.4f}·λ")
        else:
            terms.append(f"{coef:+.4f}·λ^{exp}")
    return " ".join(terms)

print("n_eff(λ) ≈", polinomio_a_str(p))

#Evalua K_lamb a partir de los coeficientes del ajuste
def k_lamb(p, lamb):
    neff = np.polyval(p,lamb)
    return (2*np.pi/lamb)*neff

def k_lambcheb(lamb):
    return (2*np.pi/lamb)*neff(lamb)

lamp = np.linspace(0.5, 1.6, 200)

k = k_lamb(p, lamp)
k2 = k_lambcheb(lamp)

#Evalya la derivada de K_lamb a partir de los coeficientes del ajuste
def kprime_lamb(p, lamb):
    neff  = np.polyval(p, lamb)
    dp    = np.polyder(p)
    dneff = np.polyval(dp, lamb)
    return (1/c)*(neff - lamb*(dneff))

def kprime_lambcheb(lamb):
    return(1/c)*(neff(lamb)-lamb*(dneff(lamb))) 

dk = kprime_lamb(p, lamp)
dk2 = kprime_lambcheb(lamp)

plt.figure(figsize=(8,5))
plt.scatter(lamp, k, label="dK/dL", color="blue", s= 5)
plt.scatter(lamp, k2, label="dK/dL", color="red", s= 5)
plt.legend()
plt.show()

n_eff(λ) ≈ -4.0113·λ^30 +22.0651·λ^29 -26.1569·λ^28 -35.6106·λ^27 +29.5098·λ^26 +100.9955·λ^25 +57.9093·λ^24 -142.6438·λ^23 -336.7339·λ^22 -188.9151·λ^21 +477.5966·λ^20 +1189.3069·λ^19 +754.0172·λ^18 -1623.4999·λ^17 -4230.8191·λ^16 -2305.3268·λ^15 +6879.2734·λ^14 +14333.3137·λ^13 +274.1838·λ^12 -34035.4062·λ^11 -29296.1571·λ^10 +63443.9807·λ^9 +92363.4656·λ^8 -151585.2446·λ^7 -155803.5389·λ^6 +517367.7363·λ^5 -540947.0876·λ^4 +309416.5522·λ^3 -103974.3708·λ^2 +19380.7368·λ -1553.2609


C:\Users\david\AppData\Local\Temp\ipykernel_29252\76551773.py:10: RankWarning: Polyfit may be poorly conditioned
  coef = np.polyfit(x, y, grado)


In [33]:
# --- Mallas espectrales ---
lamp = np.linspace(0.5, 1.6, 200)
lams = np.linspace(0.5, 1.6, 200)
LAMP, LAMS = np.meshgrid(lamp, lams)

# --- Conservación de energía ---
LAMI = 1 / (2 / LAMP - 1 / LAMS)

# --- Filtro de valores físicos ---
mask = np.isfinite(LAMI) & (LAMI > 0)

# --- Cálculos de k y Δk ---    

Kp = k_lambcheb(LAMP)
Ks = k_lambcheb(LAMS)
Ki = np.full_like(Kp, np.nan)  # para llenar solo los válidos
Ki[mask] = k_lambcheb(LAMI[mask])

DK = 2 * Kp - Ks - Ki

# --- Retardos de grupo ---
taus = np.full_like(Kp, np.nan)
taui = np.full_like(Kp, np.nan)

taus[mask] = kprime_lambcheb(LAMP[mask]) - kprime_lambcheb(LAMS[mask])
taui[mask] = kprime_lambcheb(LAMP[mask]) - kprime_lambcheb(LAMI[mask])

# --- Gráfica de condiciones de GVM ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True,
                         constrained_layout=True)

titles = [
    "Caso 1: τᵢ = 0 (idler y pump sincronizados)",
    "Caso 2: τₛ = 0 (signal y pump sincronizados)",
    "Caso 3: τₛ + τᵢ = 0 (factorabilidad)"
]
colors = ['r', 'g', 'orange']
taus_list = [taui, taus, taui + taus]

# --- Escala logarítmica segura para Δk ---
vmax = np.nanpercentile(np.abs(DK), 99)
vmin = -vmax

for ax, title, color, tau in zip(axes, titles, colors, taus_list):
    im = ax.imshow(DK, extent=[lamp.min(), lamp.max(), lams.min(), lams.max()],
                   origin='lower', aspect='auto', cmap='coolwarm',
                   alpha=0.4, vmin=vmin, vmax=vmax)

    # Contornos Δk = 0 (phase matching) y τ = 0 (GVM)
    CS1 = ax.contour(LAMP, LAMS, DK, levels=[0], colors='b', linewidths=2)
    CS2 = ax.contour(LAMP, LAMS, tau, levels=[0], colors=color, linewidths=2)

    ax.set_title(title, fontsize=11)
    ax.set_xlabel(r'$\lambda_p$ (µm)')
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

axes[0].set_ylabel(r'$\lambda_s$ (µm)')
fig.colorbar(im, ax=axes, orientation='vertical', shrink=0.8,
             label=r'$\Delta k$ [1/µm]', pad=0.02)
fig.suptitle("Condiciones de Phase Matching y Group Velocity Matching", fontsize=14)
plt.show()

In [36]:
def poly_to_str(poly: Polynomial, var="λ", fmt="{:+.6g}"):
    terms=[]
    for k, ak in enumerate(poly.coef):
        if abs(ak)<1e-14: 
            continue
        if k==0: terms.append(fmt.format(ak))
        elif k==1: terms.append(fmt.format(ak)+f"·{var}")
        else: terms.append(fmt.format(ak)+f"·{var}^{k}")
    return "".join(terms) if terms else "0"

# ===== rangos en µm (convertidos desde nm) =====
lam_p_range = (0.72, 0.89)   # 720–890 nm
lam_s_range = (1.53, 1.57)   # 1530–1570 nm
lam_i_range = (0.50, 0.68)   # 500–680 nm

# mallas para la búsqueda
grid_points = 240
lamp = np.linspace(*lam_p_range, grid_points)
lams = np.linspace(*lam_s_range, grid_points)
LAMP, LAMS = np.meshgrid(lamp, lams)
LAMI = 1.0 / (2.0/LAMP - 1.0/LAMS)
mask_i = (LAMI >= lam_i_range[0]) & (LAMI <= lam_i_range[1])

intersecciones = []  # acá acumulamos todas las soluciones de todas las geometrías
j_vals = sorted(df["j"].unique())
k_vals = sorted(df["k"].unique())

for janc in df["j"].unique():
    for kalt in df["k"].unique():
        df_geomit = df[(df["j"]==janc) & (df["k"]==kalt)]
        alto = df_geomit["Alt_Si3N4"].iloc[0]
        ancho = df_geomit["Anc_Si3N4"].iloc[0]

        chebit, neffit, dneffit = ajustar_neff_vs_lambda_cheb(df_geomit)

        def k_lambcheb(lamb):
            return (2*np.pi/lamb)*neffit(lamb)

        def kprime_lambcheb(lamb):
            return(1/c)*(neffit(lamb)-lamb*(dneffit(lamb))) 

        p_mono = chebit.convert(kind=Polynomial)
        #Generamos las combinaciones de [l_bombeo, l_signal, l_idler]. 
        lamp = np.linspace(0.72, 0.890, 200)
        lams = np.linspace(1.530, 1.570, 200)
        LAMP, LAMS = np.meshgrid(lamp, lams)
        #Restringimos lami a aquellos que cumples conservacion de energia
        LAMI_ENERGY = 1 / (2 / LAMP - 1 / LAMS)
        
        #Generamos una mascara de valores booleanos que nos permitan acceder a los valores correctos de LAMI_ENERGY
        mask = np.isfinite(LAMI_ENERGY) & (LAMI_ENERGY >= 0.5) & (LAMI_ENERGY <= 0.68)

        Kp = k_lambcheb(LAMP)
        Ks = k_lambcheb(LAMS)
        Ki = np.full_like(Kp, np.nan)# Genera una estructura de datos de igual tamaño que Kp y los rellena de np.nan
        Ki[mask] = k_lambcheb(LAMI_ENERGY[mask])# Calcula las K_lamb de los valores de LAMI validos, y Ki[mask] los inserta en las posiciones correspondientes
                                     # Esto permite que  Ki siga teniendo el mismo tamaño que antes!
        
        DK = 2 * Kp - Ks - Ki

        #Retardos de grupo
        Taus = np.full_like(Kp, np.nan)
        Taui = np.full_like(Kp, np.nan)

        Taus[mask] = kprime_lambcheb(LAMP[mask]) - kprime_lambcheb(LAMS[mask])
        Taui[mask] = kprime_lambcheb(LAMP[mask]) - kprime_lambcheb(LAMI_ENERGY[mask])

        #Creamos diferentes mascaras para cada condicion. Creamos un margen de error epsilon para cada una de ellas
        epsilon = 1e-6
        mask_1 = (np.abs(DK)<= epsilon) #Dk = 0
        mask_2 = (np.abs(Taus + Taui) <= epsilon ) #Taus = -Taui
        mask_3 = (np.abs(Taus) <= epsilon ) #Taus = 0
        mask_4 = (np.abs(Taui) <= epsilon ) #Taui = 0

        #Intersecciones
        mask_12 = mask_1 & mask_2
        mask_13 = mask_1 & mask_3
        mask_14 = mask_1 & mask_4

        #Extraemos las combinaciones de lambdas que cumplen las condiciones especificadas
        def extraer_lambdas(mask_int):
            lam_p = LAMP[mask_int]
            lam_s = LAMS[mask_int]
            lam_i = LAMI_ENERGY[mask_int]
            return lam_p, lam_s, lam_i

        lam_p_12, lam_s_12, lam_i_12 = extraer_lambdas(mask_12)
        lam_p_13, lam_s_13, lam_i_13 = extraer_lambdas(mask_13)
        lam_p_14, lam_s_14, lam_i_14 = extraer_lambdas(mask_14)

        if (len(lam_p_12) > 0) or (len(lam_p_13) > 0) or (len(lam_p_14) > 0):
            intersecciones.append({
                "j": janc,
                "k": kalt,
                "λ_p_12": lam_p_12,
                "λ_s_12": lam_s_12,
                "λ_i_12": lam_i_12,
                "λ_p_13": lam_p_13,
                "λ_s_13": lam_s_13,
                "λ_i_13": lam_i_13,
                "λ_p_14": lam_p_14,
                "λ_s_14": lam_s_14,
                "λ_i_14": lam_i_14,
                "coeficientes": p 
            })
            #Graficamos en todo el rango de los lambdass

            lamp = np.linspace(0.5, 1.57, 300)
            lams = np.linspace(0.5, 1.57, 300)
            LAMP, LAMS = np.meshgrid(lamp, lams)
            #Restringimos lami a aquellos que cumples conservacion de energia
            LAMI_ENERGY = 1 / (2 / LAMP - 1 / LAMS)

            #Generamos una mascara de valores booleanos que nos permitan acceder a los valores físicamente validos de lambda idler
            mask = np.isfinite(LAMI_ENERGY) & (LAMI_ENERGY > 0)

            Kp = k_lambcheb(LAMP)
            Ks = k_lambcheb(LAMS)
            Ki = np.full_like(Kp, np.nan)# Genera una estructura de datos de igual tamaño que Kp y los rellena de np.nan
            Ki[mask] = k_lambcheb(LAMI_ENERGY[mask])# Calcula las K_lamb de los valores de LAMI validos, y Ki[mask] los inserta en las posiciones correspondientes
                                        # Esto permite que  Ki siga teniendo el mismo tamaño que antes!
            
            DK = 2 * Kp - Ks - Ki

              #Retardos de grupo
            Taus = np.full_like(Kp, np.nan)
            Taui = np.full_like(Kp, np.nan)

            Taus[mask] = kprime_lambcheb(LAMP[mask]) - kprime_lambcheb(LAMS[mask])
            Taui[mask] = kprime_lambcheb(LAMP[mask]) - kprime_lambcheb(LAMI_ENERGY[mask])

            fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True,
                                     constrained_layout=True)
            titles = [
                "Caso 1: τᵢ = 0 (idler y pump sincronizados)",
                "Caso 2: τₛ = 0 (signal y pump sincronizados)",
                "Caso 3: τₛ + τᵢ = 0 (factorabilidad)"
            ]
            colors = ['r', 'g', 'orange']
            taus_list = [Taui, Taus, Taui + Taus]

            vmax = np.nanpercentile(np.abs(DK), 99)
            vmin = -vmax

            for ax, title, color, tau in zip(axes, titles, colors, taus_list):
                # Fondo: mapa Δk
                im = ax.imshow(DK, extent=[lamp.min(), lamp.max(), lams.min(), lams.max()],
                               origin='lower', aspect='auto', cmap='coolwarm',
                               alpha=0.4, vmin=vmin, vmax=vmax)
                # Contorno Δk = 0
                ax.contour(LAMP, LAMS, DK, levels=[0], colors='b', linewidths=2)
                # Contorno τ = 0
                ax.contour(LAMP, LAMS, tau, levels=[0], colors=color, linewidths=2)

                ax.set_title(title, fontsize=11)
                ax.set_xlabel(r'$\lambda_p$ (µm)')
                ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
                ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

            axes[0].set_ylabel(r'$\lambda_s$ (µm)')
            fig.colorbar(im, ax=axes, orientation='vertical', shrink=0.8,
                         label=r'$\Delta k$ [1/µm]', pad=0.02)
            fig.suptitle(f"Condiciones de Phase Matching y Group Velocity Matching\nanchura={janc}, altura={kalt}",
                         fontsize=14)
            plt.show()

#Mostrar resumen
print(f"✅ Se analizaron {len(intersecciones)} geometrías con intersecciones.")
total_pts = sum(len(d["λ_p_12"]) + len(d["λ_p_13"]) + len(d["λ_p_14"]) for d in intersecciones)
print(f"Total de puntos de intersección encontrados: {total_pts}")

#Ejemplo: mostrar primeras n geometrias
n=4
print(f"Primeras {n} geometrias")
for d in intersecciones[:n]:
    print(f" j={d['j']}, k={d['k']}, puntos_12={len(d['λ_p_12'])}, puntos_13={len(d['λ_p_13'])}, puntos_14={len(d['λ_p_14'])}")        

        

✅ Se analizaron 14 geometrías con intersecciones.
Total de puntos de intersección encontrados: 57
Primeras 4 geometrias
 j=3, k=3, puntos_12=1, puntos_13=1, puntos_14=1
 j=3, k=10, puntos_12=1, puntos_13=1, puntos_14=1
 j=4, k=2, puntos_12=2, puntos_13=2, puntos_14=2
 j=4, k=4, puntos_12=1, puntos_13=1, puntos_14=1


In [ ]:
def ajustar_neff_vs_lambda_cheb(df, grado=30):
    x = df["lambda"].values
    y = df["n_eff"].values

    # Ajuste estable en la base de Chebyshev; 'domain' reescala x a [-1,1]
    dom = [x.min(), x.max()]
    cheb = Chebyshev.fit(x, y, deg=grado, domain=dom)  # evita mal condicionamiento

    # Devolver objeto evaluable + derivada exacta   
    def neff(xq):   return cheb(xq)
    def dneff(xq):  return cheb.deriv()(xq)
    return cheb, neff, dneff